# Generators — Pythonic Thinking

In the previous notebook we built an iterator the hard way: one class to hold the data, another class to walk it, `__iter__` and `__next__` methods, a manual cursor and an explicit `StopIteration`. It worked — and it was tedious. Generators are Python's answer to that tedium: a **generator** is a special kind of iterator you get almost for free by writing a normal function and using the `yield` keyword instead of `return`.

Why does this matter beyond saving keystrokes? Because generators make *lazy computation* the default. A generator never builds a whole result in memory; it produces one value, pauses, waits for you to ask again, then resumes exactly where it left off — like a vending machine that dispenses one snack at a time instead of dumping the entire inventory on the floor. That single property is what lets Python programs loop over a million files, a live sensor feed, or a huge dataset without running out of RAM.

Here is the roadmap for this lesson:

1. Recall the manual iterable/iterator class pair (`mera_range` + `mera_iterator`) that generators replace.
2. Write our first generator function and observe that calling it runs **no** code until the first `next()` or `for` pull.
3. Compare `yield` with `return`: pausing versus finishing, resumable state versus one-shot results.
4. See how generators remember their position between pulls, and how they die once exhausted.
5. Rebuild `range()` as a six-line generator — `mera_range` again, but with `yield`.
6. Use **generator expressions**, the lazy twin of list comprehensions.
7. Study a practical lazy reader that yields images one at a time, then walk through the four headline benefits: easier implementation, tiny memory, infinite streams, and chained pipelines.

Key definitions. A **generator function** is any function containing `yield`; calling it returns a **generator object**, which is a full iterator. Each `next()` runs the body until the next `yield`, whose value is sent back; when the function falls off the end, the generator raises `StopIteration`. Hold onto those five sentences — the rest of the notebook is just variations on them.

# 1. The boilerplate generators replace

## 1.1 Two classes, five methods — and then there was `yield`

Before writing a single generator, look hard at the code in the next cell, because it is the *before* picture of this entire lesson. `mera_range` is the iterable: it stores `start` and `end`, and its `__iter__` method hands back a `mera_iterator`. `mera_iterator` is the walker: it holds a reference back to the iterable, returns itself from `__iter__`, and in `__next__` either raises `StopIteration` (when `start >= end`) or returns the current number and nudges the cursor forward.

Count the moving parts: two classes, five methods, one piece of shared mutable state, and an explicit end-of-sequence check. Multiply that by every lazy data source you will ever write — file readers, database cursors, API paginators — and you begin to feel the weight.

This is the gap generators close. A generator function keeps the *logic* (`for i in range(start, end): yield i`) and deletes all the scaffolding: no classes, no `__next__`, no manual `StopIteration` — Python raises it for you the moment the function returns.

Keep this class version visible as a reference point. Whenever a generator example looks almost too simple to be correct, flip back to this cell and ask: which of these method bodies is Python now writing for me?

In [1]:
# iterable
class mera_range:
    
    def __init__(self,start,end):
        self.start = start
        self.end = end
        
    def __iter__(self):
        return mera_iterator(self)
    

# iterator
class mera_iterator:
    
    def __init__(self,iterable_obj):
        self.iterable = iterable_obj
    
    def __iter__(self):
        return self
    
    def __next__(self):
        
        if self.iterable.start >= self.iterable.end:
            raise StopIteration
        
        current = self.iterable.start
        self.iterable.start+=1
        return current

## 1.2 The why: values on demand

The next cell repeats a memory experiment you met in the iterators notebook, and the gap it exposes is enormous. The list comprehension `[x for x in range(100000)]` materialises every element immediately: each Python `int` is a full object, so the list alone weighs hundreds of kilobytes — the output of `sys.getsizeof(L)` makes that concrete in the benefits section later, where the same construction measures about 800 KB. `range(10000000)` — ten million numbers — is a different animal: it costs a fixed handful of bytes (the stored output here shows `48`), because it stores only a start, a stop and a step, computing element *k* only when someone asks for it.

Generators inherit exactly this philosophy, with one upgrade: they can be *arbitrary computations*, not just number sequences. Reading ten thousand CSV files? Yield one parsed row at a time. Training a model over images on disk? Yield one image array at a time. The producing code looks like a normal function with `yield`, but the consumption pattern stays lazy — pull, process, pull again.

Rule of thumb: **build a list when you genuinely need all the values at once; build a generator when you process values one by one.** Most loops process-and-discard, which is why most loops could quietly become generators.

In [2]:
L = [x for x in range(100000)]

#for i in L:
    #print(i**2)
    
import sys
sys.getsizeof(L)

x = range(10000000)

#for i in x:
    #print(i**2)
sys.getsizeof(x)

48

# 2. Your first generator

## 2.1 `yield`: a function that hands back values without finishing

The function in the next cell, `gen_demo`, looks like any other function except for one word: `yield`. Three `yield` statements replace three `return` statements — and that one word changes the function's entire nature.

Two-phase behaviour to internalise:

1. **Calling phase.** `gen = gen_demo()` runs *nothing* inside the body. No string is produced yet. What you get back is a **generator object** — a fresh iterator sitting at the top of the function, ready to execute.
2. **Production phase.** Only when something pulls — `for i in gen`, `next(gen)`, `sum(...)` — does Python run the body up to the first `yield`, pause it, hand the yielded value out, and remember *exactly* where it stopped: line number, local variables, the works. The next pull resumes on the line after that `yield`.

So iterating `gen` prints, one pull at a time, `first statement`, `second statement`, `third statement`. After the third, no `yield` remains; control falls off the end, the generator raises `StopIteration`, and the `for` loop ends.

Think of a generator as a ticket counter that rips one ticket per customer: it does not build the audience in advance, it serves on demand. Each `yield` is one ticket handed over; each pause is the counter waiting for the next customer.

One subtle consequence: code placed *before* the first `yield` still runs on the first pull, not at call time — a fact that trips people up when they put prints or setup logic at the top of a generator.

In [3]:
def gen_demo():
    
    yield "first statement"
    yield "second statement"
    yield "third statement"

In [4]:
gen = gen_demo()

for i in gen:
    print(i)

first statement
second statement
third statement


## 2.2 `yield` versus `return`

Both keywords hand a value back to the caller, but they sit at opposite ends of the lifecycle:

| | `return` | `yield` |
| --- | --- | --- |
| Effect on the function | **Finishes** it — the frame is destroyed | **Pauses** it — the frame stays alive behind the scenes |
| How many times per call | Exactly once | Once per `yield`, potentially forever |
| Caller gets | The value, then the function is gone | One value per pull; the function resumes later |
| End of output | Control returns immediately | `StopIteration` once no `yield` is left |
| What is remembered | Nothing after return | Locals, loop position, line number |

A useful analogy: `return` is *mailing* a letter — you write it, send it, and it leaves your desk for good. `yield` is *bookmarking* a book and closing it — the book (your function's state) stays on the shelf exactly where you left off, ready for the next reader, again and again.

The empty cell below is a placeholder in this notebook for a Python Tutor visualisation. Watch, when you try it yourself, how a generator's frame reappears and disappears between pulls while a normal function's frame vanishes forever after a single `return`. And reread the `square` example later in this notebook with that picture in mind: it contains a `for` loop and a `yield`, so a single call builds no list of squares — each `next()` advances the loop by one step and freezes it again. A `return i**2` in the same place would return only the first square and then terminate.

## 2.3 Generators remember where they stopped

The next two cells are the clearest proof that a generator is a *stateful* iterator, not a function you re-run from the top.

`square(10)` is a generator function: calling it returns a generator without computing a single square. Then the notebook pulls three values explicitly:

```python
print(next(gen))   # 1
print(next(gen))   # 4
print(next(gen))   # 9
```

Each call runs the loop body forward by one iteration and freezes it again. Crucially, the `for i in range(1, num + 1)` inside the generator did **not** restart — the generator remembers its loop variable between pulls. When the later `for i in gen:` loop runs, it continues with `16, 25, 36 ... 100`, which is exactly what the stored output shows: `1, 4, 9` from the explicit pulls, then `16` through `100` from the loop. One object, one continuous walk — the same behaviour as the `list_iterator` from the iterators notebook, expressed in ten lines of plain function code.

Two lifecycle rules follow:

- **Generators are single-pass.** Once the body finishes, the generator is exhausted; a further `next()` raises `StopIteration`. To walk the data again, call `square(10)` a second time to build a fresh generator.
- **Generators are cold until pulled.** Nothing inside the function body has executed at call time — you can even create a generator and never consume it, and it will cost almost nothing.

In [5]:
def square(num):
    for i in range(1,num+1):
        yield i**2

In [6]:
gen = square(10)

print(next(gen))
print(next(gen))
print(next(gen))

for i in gen:
    print(i)

1
4
9
16
25
36
49
64
81
100


# 3. `range()` again — now in four lines

## 3.1 The class pair collapses into one function

Remember `mera_range` from the iterators notebook: two classes, five methods, a manual `StopIteration` and a shared cursor? The next cell rebuilds the same idea as a generator:

```python
def mera_range(start, end):
    for i in range(start, end):
        yield i
```

That is the entire implementation. Walk through what Python now handles for you:

- Calling `mera_range(15, 26)` returns a generator — which is already an iterator, so the function object itself plays the role of the iterable.
- The loop variable `i` replaces the hand-written cursor; the built-in `range` does the counting.
- When the loop finishes, falling off the end of the function raises `StopIteration` automatically — no explicit check, no handwritten exception.
- Each `yield` freezes `i`'s value *and* the loop's progress, so consecutive pulls produce `15`, `16`, `17`, ... exactly like the class version.

The demo cell then loops over `mera_range(15, 26)` and prints `15` through `25` — the same stream the class version would produce, in roughly one-tenth of the code. Note the name reuse: `mera_range` here is a plain function, while earlier it was a class. A name is just a label; what matters is the protocol the object it produces honours.

In [7]:
def mera_range(start,end):
    
    for i in range(start,end):
        yield i

In [8]:
for i in mera_range(15,26):
    print(i)

15
16
17
18
19
20
21
22
23
24
25


# 4. Generator expressions

## 4.1 Brackets versus parentheses

A generator expression is to a generator what a list comprehension is to a list — except it builds nothing. Compare the two neighbouring cells:

```python
L = [i**2 for i in range(1, 101)]   # eager: all 100 squares exist now
gen = (i**2 for i in range(1, 101)) # lazy: one small iterator object
```

Only the brackets change: `[]` builds a list immediately, `()` builds a generator that computes each square the moment someone asks. The loop that follows prints `1, 4, 9, ...` up to `10000` in either case; the difference is *when* the work happens and how much memory it costs — hundreds of kilobytes for the list, a couple of hundred bytes for the generator object.

Properties worth memorising:

- **One-shot.** A generator expression can be consumed only once. `sum(gen)` takes everything; a second pass yields nothing.
- **Bare when sole argument:** `sum(i**2 for i in range(101))` is legal and idiomatic — the parentheses are implied.
- **Composable.** Generators feed generators without temporary lists, which is the engine behind the chaining example at the end of this notebook.
- **Same rules, fewer words.** Multiple `for` and `if` clauses work exactly as in comprehensions; only the eagerness changes.

Rule of thumb: reach for a list when you need indexing, `len()`, sorting or repeated re-use; reach for a generator expression when you will loop once and discard.

In [9]:
# list comprehension
L = [i**2 for i in range(1,101)]

In [10]:
gen = (i**2 for i in range(1,101))

for i in gen:
    print(i)

1
4
9
16
25
36
49
64
81
100
121
144
169
196
225
256
289
324
361
400
441
484
529
576
625
676
729
784
841
900
961
1024
1089
1156
1225
1296
1369
1444
1521
1600
1681
1764
1849
1936
2025
2116
2209
2304
2401
2500
2601
2704
2809
2916
3025
3136
3249
3364
3481
3600
3721
3844
3969
4096
4225
4356
4489
4624
4761
4900
5041
5184
5329
5476
5625
5776
5929
6084
6241
6400
6561
6724
6889
7056
7225
7396
7569
7744
7921
8100
8281
8464
8649
8836
9025
9216
9409
9604
9801
10000


# 5. A practical generator: reading images lazily

## 5.1 One image in memory at a time

Machine-learning folders are a perfect generator use case: a training folder can hold thousands of images, and decoding them all into RAM *before* training starts is a fast route to an out-of-memory crash. The function in the next cell flips that script:

```python
def image_data_reader(folder_path):
    for file in os.listdir(folder_path):
        f_array = cv2.imread(os.path.join(folder_path, file))
        yield f_array
```

Read it as a patient conveyor belt. `os.listdir` merely *names* the files (cheap); `cv2.imread` decodes **one** file into a NumPy array; `yield` hands that array to the consumer and freezes the loop. The next pull decodes the next file. At any instant only a single decoded image exists inside the generator — everything else is still on disk as compressed file bytes.

Because a generator is an iterator, the consumer controls the pace. The demonstration cell pulls a few frames manually with `next(gen)` — each call reads exactly one more image. The same generator would work inside a `for` loop, or be wrapped in batching logic that yields chunks for a data loader.

Two practical notes. First, this cell depends on OpenCV (`cv2`) being installed. Second, it points at a fixed folder path from the original author's machine — on your computer, replace it with a path that actually contains images; any folder of `.jpg` files will do. The pattern, not the hard-coded path, is the lesson.

In [11]:
import os
import cv2

def image_data_reader(folder_path):

    for file in os.listdir(folder_path):
        f_array = cv2.imread(os.path.join(folder_path,file))
        yield f_array
    

In [12]:
gen = image_data_reader('C:/Users/91842/emotion-detector/train/Sad')

next(gen)
next(gen)

next(gen)
next(gen)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/91842/emotion-detector/train/Sad'

# 6. Benefits of using a generator

Four claims are usually made for generators, and this notebook demonstrates each one with actual code rather than slogans. Think of a generator as a delivery service that carries one package at a time along a route: cheap to set up, light on the vehicle, able to serve routes that never end, and easy to hand over to the next driver. In the language of Python, those four qualities have concrete names and concrete proofs.

First, a generator is far **easier to implement** than the equivalent iterator class pair — the next-but-one cell shows the two designs side by side. Second, it is dramatically **memory efficient**, holding only a paused frame instead of a full collection — the size comparison that follows is the single most persuasive number in this notebook. Third, it can represent **infinite streams** that lists could never store — an endless source of even numbers costs nothing. Fourth, generators **chain** into pipelines where values flow through several transformations without ever landing in an intermediate list — fib, squared, summed. The next four sections take each benefit one at a time, with a small experiment to back the claim.

## 6.1 Ease of implementation

Side by side, the comparison is brutal. The class version needs `mera_range` — a data holder with `__iter__` — plus `mera_iterator`, a cursor with `__iter__` and `__next__`: two classes, five methods, an explicit `start >= end` check and a manual `raise StopIteration`. The generator version is the four-line function you have already seen: a `for` loop and a single `yield`.

Everything in between is scaffolding you no longer maintain — no cursor attribute to keep in sync, no risk of forgetting `return self` in `__iter__`, no end-of-sequence logic to duplicate per iterator type. Python generates a tuned iterator object for you, tracks its position internally and raises `StopIteration` at exactly the right moment.

Ease here is not cosmetic. Fewer lines means fewer places for off-by-one bugs (`< end` versus `<= end` is a classic), straightforward testing — the body is an ordinary function — and readable intent: the word `yield` states *'produce these values over time'* on sight. When a four-line function fully replaces a two-class protocol, that is the language telling you which shape it prefers.

In [13]:
class mera_range:
    
    def __init__(self,start,end):
        self.start = start
        self.end = end
        
    def __iter__(self):
        return mera_iterator(self)

In [14]:
# iterator
class mera_iterator:
    
    def __init__(self,iterable_obj):
        self.iterable = iterable_obj
    
    def __iter__(self):
        return self
    
    def __next__(self):
        
        if self.iterable.start >= self.iterable.end:
            raise StopIteration
        
        current = self.iterable.start
        self.iterable.start+=1
        return current

In [15]:
def mera_range(start,end):
    
    for i in range(start,end):
        yield i

## 6.2 Memory efficient

The measurement cell prints the punchline of this entire notebook:

```text
Size of L in memory 800984
Size of gen in memory 200
```

Same hundred thousand integers — roughly **800 KB** as a list versus **200 bytes** as a generator. That is not a rounding difference; it is about four thousand times smaller, and the gap only grows with the data. A hundred million rows would be gigabytes as a list yet still a few hundred bytes as a generator object.

Where does the tiny footprint come from? A list stores every element — object headers, reference pointers, the works. A generator stores only its *paused frame*: the current value of the loop variable, the iterator it is reading from, and the line number to resume at. Elements are created, handed to you, and dropped; the generator never collects them.

Two caveats keep the claim honest. First, the 200 bytes measure the generator *object*, not the individual values flowing through it — memory stays flat, but each yielded value still exists while you hold it. Second, laziness only helps if you genuinely process-and-discard; collecting everything afterwards with `results = list(gen)` simply moves the big allocation into your own variable.

In [16]:
L = [x for x in range(100000)]
gen = (x for x in range(100000))

import sys

print('Size of L in memory',sys.getsizeof(L))
print('Size of gen in memory',sys.getsizeof(gen))

Size of L in memory 800984
Size of gen in memory 200


## 6.3 Representing infinite streams

Some sequences never end: sensor readings, counters, timestamps, the tail of a live log. A list cannot represent any of these — *all even numbers* has no final element, so the container can never finish building. A generator can, because it only ever promises the *next* value:

```python
def all_even():
    n = 0
    while True:
        yield n
        n += 2
```

`while True` plus `yield` is the signature of an infinite generator. The demo cell creates `even_num_gen = all_even()` and pulls twice with `next()`, receiving `0` and `2`; the generator then sits paused at `n = 4`, ready indefinitely, and its memory use stays constant no matter how many values are consumed.

The flip side is discipline. Never drop an infinite generator into an ordinary `for` loop — `StopIteration` will never arrive, so the loop would run until you interrupt the kernel. Consume infinite streams deliberately: a fixed number of `next()` calls, a slice from `itertools.islice(gen, 100)`, or a `zip` with a finite partner that ends first. In other words, infinity is safe only when something else decides when to stop.

In [17]:
def all_even():
    n = 0
    while True:
        yield n
        n += 2

In [18]:
even_num_gen = all_even()
next(even_num_gen)
next(even_num_gen)

2

## 6.4 Chaining generators

The final cell is the payoff — three functions acting as three stations on one assembly line:

```python
sum(square(fibonacci_numbers(10)))
```

- `fibonacci_numbers(10)` yields ten Fibonacci numbers with a clean tuple-swap: `1, 1, 2, 3, 5, 8, 13, 21, 34, 55`.
- `square(...)` receives those values one at a time and yields each one squared.
- `sum(...)` pulls from `square`, which pulls from `fibonacci_numbers`, and totals everything: `4895`.

No intermediate list exists anywhere in that chain. Because every stage is a generator, each value is born at `fibonacci_numbers`, transformed in `square`, and consumed by `sum` — then all three forget about it. This *pull-based pipeline* is the backbone of a great deal of Python data work: `itertools`, lazy engines, and large file-processing scripts are all variations on this shape.

Two practical dividends. You can insert a new stage — a filter, a log, a batching step — by wrapping one more generator around the chain without rewriting anything. And memory stays flat no matter how many values flow through, because only one value occupies each stage at a time. When you catch yourself building throwaway lists in `for` loops, reach for a generator chain instead.

In [19]:
def fibonacci_numbers(nums):
    x, y = 0, 1
    for _ in range(nums):
        x, y = y, x+y
        yield x

def square(nums):
    for num in nums:
        yield num**2

print(sum(square(fibonacci_numbers(10))))

4895


## Summary

A **generator** is an iterator you get by writing a normal function that uses `yield` instead of `return`. Calling that function produces a cold generator object; each pull runs the body to the next `yield`, freezes the local state, and hands over one value; when no `yield` remains, Python raises `StopIteration` for you. Generators beat hand-written iterator classes on every axis shown here: four lines instead of two classes, about 200 bytes instead of about 800 KB for the same hundred thousand values, the ability to model infinite streams with `while True`, and the freedom to chain stages — `sum(square(fibonacci_numbers(10)))` — so data flows without ever sitting in a list. Generator expressions, parentheses instead of brackets, give the same laziness in one line. Reach for a generator when you process values one at a time; use a list when you truly need them all at once.